# Detecção de fraude em rastreadores veiculares: treinamento e avaliação

Fluxo completo do trabalho, do arquivo bruto de auditoria até os modelos finais:

1. tratamento das planilhas de auditoria (opcional, se os *_treated.csv já existem);
2. montagem da base em janelas de 60 minutos por rastreador;
3. rótulo de fraude: jammer persistente e/ou violação de alimentação, ou períodos confirmados;
4. comparação de Regressão Logística, Rede Neural (MLP) e XGBoost em dois protocolos:
   - **A, temporal**: treino no período inicial, teste no período final;
   - **B, entre rastreadores**: StratifiedGroupKFold, nenhum rastreador do teste aparece no treino;
5. decisão final por rastreador (indícios combinados);
6. figuras para a monografia em resultados/figuras e modelos finais em resultados/modelos_finais.pkl.

O código reutilizável está no pacote fraude/ deste repositório.

## Preparação do ambiente

In [ ]:
import os, sys

REPO = "https://github.com/Dapalha/projeto-monografia-regressao.git"
PASTA_REPO = "/content/projeto-monografia-regressao"

if "google.colab" in sys.modules:
    if not os.path.exists(PASTA_REPO):
        !git clone -q {REPO} {PASTA_REPO}
    %cd {PASTA_REPO}
    !pip -q install -r requirements.txt
    from google.colab import drive
    drive.mount("/content/drive")

sys.path.insert(0, os.getcwd())

In [ ]:
import gc
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit, StratifiedGroupKFold

from fraude import features as ft
from fraude import figuras as fig
from fraude.avaliacao import avaliar_decisao, imprimir, metricas
from fraude.base import aplicar_fraudes_confirmadas, montar_base
from fraude.modelos import MODELOS, SIGLA, limiar_por_precisao, treinar
from fraude.tratamento import COLUNAS_DUPLICADAS, LINHAS_CORRIGIDAS, tratar_arquivo

DRIVE = Path("/content/drive/MyDrive")
DIR_BRUTOS = DRIVE / "dados_brutos"          # planilhas de auditoria originais
DIR_TRATADOS = DRIVE / "dataset_tratado"     # *_treated.csv
DIR_RESULT = DIR_TRATADOS / "resultados"
DIR_FIG = DIR_RESULT / "figuras"
DIR_FIG.mkdir(parents=True, exist_ok=True)

TRATAR_DADOS_BRUTOS = False   # True para gerar os *_treated.csv a partir de dados_brutos/
RECONSTRUIR_BASE = False      # True depois de alterar ft.PARAMS ou os arquivos tratados
PRECISAO_MINIMA = 0.5         # precisão mínima exigida na escolha do limiar
PESO_ENERGIA = 3.0            # peso das variáveis de alimentação no XGBoost
N_REPETICOES = 3              # repetições da validação cruzada por grupos
SEED = 42

fig.estilo()

## 1. Tratamento dos dados brutos

In [ ]:
if TRATAR_DADOS_BRUTOS:
    DIR_TRATADOS.mkdir(parents=True, exist_ok=True)
    arquivos = sorted([*DIR_BRUTOS.glob("*.csv"), *DIR_BRUTOS.glob("*.xlsx")])
    falhas = []
    for fp in arquivos:
        try:
            df, _ = tratar_arquivo(fp)
            destino = DIR_TRATADOS / f"{fp.stem.replace('_auditoria', '')}_treated.csv"
            df.to_csv(destino, sep=";", index=False, encoding="utf-8")
        except Exception as e:
            falhas.append((fp.name, f"{type(e).__name__}: {e}"))
    print(f"{len(arquivos) - len(falhas)} de {len(arquivos)} arquivos tratados")
    print(f"linhas com ';' extra reagrupadas: {sum(LINHAS_CORRIGIDAS.values())}")
    if COLUNAS_DUPLICADAS:
        print("cabeçalhos com nomes repetidos:", COLUNAS_DUPLICADAS)
    for nome, erro in falhas:
        print(f"  {nome}: {erro}")

## 2. Base de janelas

In [ ]:
cache = DIR_RESULT / "base_janelas.pkl"
if cache.exists() and not RECONSTRUIR_BASE:
    base, dur_jammer = pd.read_pickle(cache)
else:
    base, dur_jammer = montar_base(sorted(DIR_TRATADOS.glob("*_treated.csv")))
    pd.to_pickle((base, dur_jammer), cache)

confirmadas = DIR_RESULT / "fraudes_confirmadas.csv"
if confirmadas.exists():
    n = aplicar_fraudes_confirmadas(base, confirmadas)
    print(f"rótulo: {n} períodos de fraude confirmados")
else:
    print(f"rótulo automático ({ft.PARAMS['ROTULO_MODO']}): "
          f"{int(base['_rot_jammer'].sum())} janelas com jammer persistente, "
          f"{int(base['_rot_energia'].sum())} com violação de alimentação")

por_rastreador = base.groupby("rastreador")[ft.TARGET].max()
print(f"{len(base)} janelas | {base['rastreador'].nunique()} rastreadores | "
      f"{int(base[ft.TARGET].sum())} janelas com fraude ({base[ft.TARGET].mean():.2%}) | "
      f"{int(por_rastreador.sum())} rastreadores com fraude")

### Duração dos episódios de jammer

In [ ]:
dur = pd.Series(dur_jammer, dtype=float)
print(dur.describe(percentiles=[.5, .9, .99]).round(1))
for limite in [5, 15, 30, 60, 120]:
    print(f">= {limite:3d} min: {(dur >= limite).sum()} episódios")

## 3. Matriz de variáveis

In [ ]:
FEATURES = ft.lista_features(base)
X = base[FEATURES].to_numpy(dtype=np.float32)
X[~np.isfinite(X)] = np.nan
y = base[ft.TARGET].to_numpy(dtype=np.int8)
grupos = base["rastreador"].astype(str).to_numpy()
ts = base["inicio_janela"].to_numpy("datetime64[ns]")
print(f"X: {X.shape[0]} janelas x {X.shape[1]} variáveis")

## 4. Experimento A: generalização temporal
Treino nos 60% iniciais do período, validação (escolha do limiar) nos 20% seguintes e teste nos 20% finais.

In [ ]:
c60, c80 = np.quantile(ts.astype("int64"), [0.6, 0.8]).astype("datetime64[ns]")
tr_A = np.where(ts < c60)[0]
va_A = np.where((ts >= c60) & (ts < c80))[0]
te_A = np.where(ts >= c80)[0]
for nome, idx in [("treino", tr_A), ("validação", va_A), ("teste", te_A)]:
    print(f"{nome:10s} {len(idx):7d} janelas, {int(y[idx].sum())} com fraude")

res_A, prob_A = [], {}
for nome in MODELOS:
    modelo = treinar(nome, X, y, tr_A, FEATURES, PESO_ENERGIA, SEED)
    limiar = limiar_por_precisao(y[va_A], modelo.predict_proba(X[va_A])[:, 1], PRECISAO_MINIMA)
    prob_A[nome] = modelo.predict_proba(X[te_A])[:, 1]
    m = metricas(y[te_A], prob_A[nome], limiar, grupos[te_A])
    imprimir(nome, "A", m)
    res_A.append({"modelo": nome, **m})
    del modelo
    gc.collect()

res_A = pd.DataFrame(res_A).set_index("modelo")
res_A.to_csv(DIR_RESULT / "experimento_A.csv", sep=";")

## 5. Experimento B: generalização entre rastreadores
Cada rastreador fica inteiro em um único fold. Em cada fold, 25% dos rastreadores de treino são separados para escolher o limiar. A divisão é repetida com sementes diferentes para reduzir o efeito da sorte na distribuição das fraudes.

In [ ]:
vazio = np.zeros((len(y), 1))
n_folds = min(5, len(np.unique(grupos)))
oof = {nome: np.full(len(y), np.nan) for nome in MODELOS}   # previsões fora da amostra (1ª repetição)
res_B, composicao = [], []

for rep in range(N_REPETICOES):
    cv = StratifiedGroupKFold(n_splits=n_folds, shuffle=True, random_state=SEED + rep)
    for fold, (tr_idx, te_idx) in enumerate(cv.split(vazio, y, grupos)):
        gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=SEED + rep)
        a, b = next(gss.split(vazio[tr_idx], y[tr_idx], grupos[tr_idx]))
        tr_i, va_i = tr_idx[a], tr_idx[b]
        assert not set(grupos[tr_i]) & set(grupos[te_idx])

        fraudes = pd.Series(y[te_idx]).groupby(grupos[te_idx]).sum()
        composicao.append(dict(rep=rep, fold=fold, rastreadores=len(fraudes),
                               com_fraude=int((fraudes > 0).sum()), janelas=len(te_idx),
                               janelas_fraude=int(fraudes.sum()),
                               concentracao=fraudes.max() / max(fraudes.sum(), 1)))

        for nome in MODELOS:
            modelo = treinar(nome, X, y, tr_i, FEATURES, PESO_ENERGIA, SEED + rep)
            limiar = limiar_por_precisao(y[va_i], modelo.predict_proba(X[va_i])[:, 1], PRECISAO_MINIMA)
            prob = modelo.predict_proba(X[te_idx])[:, 1]
            if rep == 0:
                oof[nome][te_idx] = prob
            res_B.append({"modelo": nome, "rep": rep, "fold": fold,
                          **metricas(y[te_idx], prob, limiar, grupos[te_idx])})
            del modelo
            gc.collect()
    print(f"repetição {rep + 1}/{N_REPETICOES} concluída")

res_B = pd.DataFrame(res_B)
composicao = pd.DataFrame(composicao)
res_B.to_csv(DIR_RESULT / "experimento_B_folds.csv", sep=";", index=False)
composicao.to_csv(DIR_RESULT / "composicao_folds.csv", sep=";", index=False)

cols = ["precisao", "recall", "f1", "roc_auc", "pr_auc", "falsos_por_rastreador_mes"]
media_B = res_B.groupby("modelo")[cols].mean().reindex(MODELOS)
desvio_B = res_B.groupby("modelo")[cols].std().reindex(MODELOS)
media_B.round(3).astype(str).add(" ± ").add(desvio_B.round(3).astype(str))

### Composição dos folds
"concentracao" é a fração das janelas de fraude do fold que vem de um único rastreador. Valores altos explicam boa parte da variação entre folds.

In [ ]:
composicao[composicao["rep"] == 0]

## 6. Decisão final por rastreador
Aplica a mesma regra usada no classificador: um episódio só é fraude se reunir pelo menos ft.DECISAO["MIN_EVIDENCIAS"] categorias de indício e a confiança combinada atingir ft.DECISAO["RISCO_MIN"]. Usa as previsões fora da amostra do experimento B.

In [ ]:
decisao, resumo_decisao = {}, {}
for nome in MODELOS:
    decisao[nome], resumo_decisao[nome] = avaliar_decisao(base, oof[nome], ft.PARAMS, ft.DECISAO)
resumo_decisao = pd.DataFrame(resumo_decisao).T
resumo_decisao.to_csv(DIR_RESULT / "decisao_por_rastreador.csv", sep=";")
resumo_decisao.round(3)

## 7. Figuras para a monografia
Uma imagem por modelo e por métrica, salvas em resultados/figuras.

In [ ]:
for nome in MODELOS:
    s = SIGLA[nome]
    fig.matriz_confusao(res_A.loc[nome], f"{nome}: teste temporal",
                        DIR_FIG / f"{s}_matriz_temporal.png")
    soma_B = res_B[res_B["rep"] == 0].groupby("modelo")[["VP", "FN", "FP", "VN"]].sum().loc[nome]
    fig.matriz_confusao(soma_B, f"{nome}: rastreadores não vistos",
                        DIR_FIG / f"{s}_matriz_grupos.png")
    fig.matriz_confusao(resumo_decisao.loc[nome], f"{nome}: decisão por rastreador",
                        DIR_FIG / f"{s}_matriz_rastreadores.png")
    fig.curva_roc(y[te_A], {nome: prob_A[nome]}, f"Curva ROC: {nome}", DIR_FIG / f"{s}_roc.png")
    fig.curva_pr(y[te_A], {nome: prob_A[nome]}, f"Curva precisão-recall: {nome}", DIR_FIG / f"{s}_pr.png")

In [ ]:
fig.curva_roc(y[te_A], prob_A, "Curvas ROC (teste temporal)", DIR_FIG / "comparativo_roc.png")
fig.curva_pr(y[te_A], prob_A, "Curvas precisão-recall (teste temporal)", DIR_FIG / "comparativo_pr.png")
fig.barras_metricas(res_A.reindex(MODELOS), "Métricas no teste temporal",
                    DIR_FIG / "comparativo_metricas_temporal.png")
fig.barras_metricas(media_B, "Métricas em rastreadores não vistos (média ± desvio)",
                    DIR_FIG / "comparativo_metricas_grupos.png", desvio=desvio_B)
fig.barras_metricas(resumo_decisao.reindex(MODELOS), "Decisão final por rastreador",
                    DIR_FIG / "comparativo_decisao_rastreadores.png",
                    metricas=("precisao", "recall", "f1"))
for metrica in ["pr_auc", "f1", "precisao", "recall"]:
    fig.boxplot_folds(res_B, metrica, f"Variação entre folds: {fig.ROTULOS[metrica]}",
                      DIR_FIG / f"folds_{metrica}.png")
fig.composicao_folds(composicao, DIR_FIG / "composicao_folds.png")

## 8. Modelos finais
Cada modelo é treinado com toda a base e salvo junto com as variáveis, os parâmetros e a regra de decisão usados no classificador.

In [ ]:
idx_total = np.arange(len(y))
finais = {}
for nome in MODELOS:
    modelo = treinar(nome, X, y, idx_total, FEATURES, PESO_ENERGIA, SEED)
    limiar = float(res_B.loc[res_B["modelo"] == nome, "limiar"].median())
    finais[nome] = {"modelo": modelo, "limiar": limiar}
    fig.importancia(modelo, nome, FEATURES, X, y, DIR_FIG / f"{SIGLA[nome]}_importancia.png")

pacote = {"modelos": finais, "features": FEATURES, "params": ft.PARAMS,
          "decisao": ft.DECISAO, "precisao_minima": PRECISAO_MINIMA}
joblib.dump(pacote, DIR_RESULT / "modelos_finais.pkl")
print("modelos salvos em", DIR_RESULT / "modelos_finais.pkl")

## 9. Tabela de métricas para a monografia

In [ ]:
tabela = pd.concat({
    "Temporal": res_A.reindex(MODELOS)[["precisao", "recall", "f1", "pr_auc", "roc_auc"]],
    "Entre rastreadores": media_B[["precisao", "recall", "f1", "pr_auc", "roc_auc"]],
    "Por rastreador": resumo_decisao.reindex(MODELOS)[["precisao", "recall", "f1"]],
}, axis=1).round(3)
tabela.to_csv(DIR_RESULT / "tabela_metricas.csv", sep=";")
tabela